# R1: Verify the Curated Dataset

This notebook confirms that the training dataset you downloaded is intact and
has the structure the paper relies on. It runs read-only (`DRY_RUN = True`);
nothing is moved or deleted.

Results for the released `Dataset_2014_2015`:

| Check | Expected result |
|-------|-----------------|
| Image counts | No Eruptions 46,517, Small 6,909, Large 645 |
| Cross-class duplicate filenames | 0 |
| Large-eruption events (10 s gap) | 129, all exactly 5 frames |
| Small-eruption events (10 s gap) | 2,303, all exactly 3 frames |
| No-eruption events (10 s gap) | 17,008 |
| No-eruption events (300 s gap) | 3,833 |
| Cross-class proximity conflicts (<=10 s) | 0 |

The event counts tie back to the paper's provenance: 129 large events =
115 clean + 14 recovered during denoising; the 1,288 large-class images =
arithmetic of 14 events x 5 frames + 406 x 3 is checked in notebook P3.

In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()

In [ ]:
import os
import glob
import pandas as pd

DRY_RUN = True   # read-only; set False only if you intend to modify the dataset

ROOT = config.DATASET_DIR
assert ROOT.exists(), (
    f"Dataset not found at {ROOT}. Download Dataset_2014_2015 from the Zenodo "
    "record and unzip it there (see README), or edit DATASET_DIR in config.py.")


## 1. Image counts and cross-class duplicate check

In [ ]:
file_sets = {}
for label, folder in config.CLASS_FOLDERS.items():
    fs = {os.path.basename(f) for f in glob.glob(str(ROOT / folder / "*.jpg"))}
    file_sets[folder] = fs
    print(f"{folder:16s}: {len(fs):,} images")

print("\nCross-class duplicate filenames (should all be 0):")
folders = list(config.CLASS_FOLDERS.values())
for i in range(len(folders)):
    for j in range(i + 1, len(folders)):
        n = len(file_sets[folders[i]] & file_sets[folders[j]])
        flag = "" if n == 0 else "  <-- INVESTIGATE"
        print(f"  {folders[i]} n {folders[j]}: {n}{flag}")


## 2. Event grouping and length integrity

Images are grouped into events by timestamp gaps. Eruption frames sit ~2 s
apart, so a gap > `EVENT_GAP_SEC` (10 s) starts a new event. Every large event
should contain exactly 5 frames and every small event exactly 3: that is the
temporal-composite structure the model was trained on.

In [ ]:
def timestamp_of(filename):
    stem = os.path.splitext(filename)[0].split('_')
    return pd.to_datetime(stem[0] + ' ' + stem[1], format='%Y-%m-%d %H%M%S',
                          errors='coerce')

records = []
for label, folder in config.CLASS_FOLDERS.items():
    for name in file_sets[folder]:
        records.append({'name': name, 'label': label, 'folder': folder,
                        't': timestamp_of(name)})
df = pd.DataFrame(records).dropna(subset=['t']).sort_values('t').reset_index(drop=True)

def event_group_sizes(sub, gap_sec):
    sub = sub.sort_values('t').reset_index(drop=True)
    new_event = (sub['t'].diff() > pd.Timedelta(seconds=gap_sec)) | sub['t'].diff().isna()
    return sub.groupby(new_event.cumsum()).size()

for folder, expected in [("Large Eruptions", 5), ("Small Eruptions", 3)]:
    sub = df[df['folder'] == folder]
    sizes = event_group_sizes(sub, config.EVENT_GAP_SEC)
    irregular = sizes[sizes != expected]
    print(f"{folder}: {sizes.shape[0]} events (gap {config.EVENT_GAP_SEC} s); "
          f"irregular-length events: {len(irregular)}")
    print(f"   size distribution: {dict(sizes.value_counts().sort_index())}")


## 3. No-eruption event count at the background scale

Background frames are grouped at `NO_ERUPTION_GROUPING_SEC` (300 s) because the
lava-lake surface cycles slowly and frames minutes apart are near-duplicates.
This is the count that keeps near-identical background in the same
train/validation/test (held-out) fold.

In [ ]:
sub = df[df['folder'] == "No Eruptions"]
for gap in (config.EVENT_GAP_SEC, config.NO_ERUPTION_GROUPING_SEC):
    n = event_group_sizes(sub, gap).shape[0]
    print(f"No Eruptions events at {gap:>3d} s gap: {n:,}")


## 4. Cross-class temporal proximity

No two frames of *different* classes should sit within `EVENT_GAP_SEC` of each
other; if they did, one physical moment would carry two labels and leak between
folds. The released dataset has already been cleaned, so this should report 0.

In [ ]:
df['dt_prev'] = df['t'].diff().dt.total_seconds()
df['prev_label'] = df['label'].shift(1)
conflicts = df[(df['dt_prev'] <= config.EVENT_GAP_SEC) &
               (df['label'] != df['prev_label'])]
print(f"Cross-class proximity conflicts (<= {config.EVENT_GAP_SEC} s apart, "
      f"different class): {len(conflicts)}")
if len(conflicts):
    print(conflicts[['name', 'folder', 'dt_prev']].head(20).to_string())
else:
    print("None -- dataset is temporally clean.")


## Summary

If the four checks above match the expected table in the header, the dataset is
the exact curated set used in the paper and you can proceed to **R2** (retrain
the model) or **R3** (evaluate the shipped model).